# Preparación completa de datasets — notebook autónomo

Este notebook contiene todo el código de procesamiento y usa el repositorio de GitHub como directorio del proyecto. En Colab clona automáticamente el repositorio público, sin depender de Google Drive. Puede ejecutarse mediante **Entorno de ejecución → Ejecutar todas**.

Genera:

- Dataset A: OPUS-100 español→inglés para fine-tuning NMT.
- Dataset B: 20 audios FLEURS con transcripción española y referencia inglesa oficial.
- Manifiestos, estadísticas y reportes de validación.

In [1]:
# 1. Instalar dependencias (reiniciar el entorno no debería ser necesario)
%pip install --quiet "datasets>=2.20.0" "pandas>=2.0.0" "soundfile>=0.12.0" "requests>=2.31.0"

In [2]:
# 2. Configuración y directorios
import os
import shutil
import subprocess
import unicodedata
import zipfile
from io import BytesIO, StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import soundfile as sf
from datasets import Audio, load_dataset
from IPython.display import display

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
REPO_URL = "https://github.com/Setrox1250/Proyecto-IA-Traductor.git"
REPO_DIR = Path("/content/Proyecto-IA-Traductor")
if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    PROJECT_ROOT = REPO_DIR
else:
    PROJECT_ROOT = Path.cwd().resolve()

TRANSLATION_DIR = PROJECT_ROOT / "data/processed/translation"
MANIFEST_DIR = PROJECT_ROOT / "data/processed/manifests"
AUDIO_DIR = PROJECT_ROOT / "data/raw/audio"
REPORT_DIR = PROJECT_ROOT / "outputs/dataset"
for directory in (TRANSLATION_DIR, MANIFEST_DIR, AUDIO_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_SIZE = 100_000
N_AUDIO_SAMPLES = 20
SEED = 42
DEV_RATIO = 0.70
OPUS_SHUFFLE_BUFFER = 10_000
FLEURS_SHUFFLE_BUFFER = 100

print("Directorio de trabajo:", PROJECT_ROOT)
print({"train_pairs": TRAIN_SIZE, "audio_samples": N_AUDIO_SAMPLES, "seed": SEED})

Directorio de trabajo: /content/Proyecto-IA-Traductor
{'train_pairs': 100000, 'audio_samples': 20, 'seed': 42}


## Dataset A — OPUS-100 para fine-tuning NMT

Se usa la configuración `en-es` y se lee en streaming para limitar el consumo de RAM. Se conservan los splits oficiales, se normalizan Unicode y espacios, se eliminan vacíos y duplicados exactos y se evita solapamiento monolingüe entre entrenamiento y los conjuntos reservados.

In [3]:
# 3. Funciones de limpieza del corpus paralelo
def normalize_text(value):
    if value is None:
        return ""
    return " ".join(unicodedata.normalize("NFC", str(value)).split())


def extract_pair(row):
    translation = row.get("translation") or {}
    return normalize_text(translation.get("es")), normalize_text(translation.get("en"))


def clean_rows(rows, limit=None, forbidden_es=None, forbidden_en=None):
    forbidden_es = forbidden_es or set()
    forbidden_en = forbidden_en or set()
    seen_pairs = set()
    output = []
    for row in rows:
        text_es, text_en = extract_pair(row)
        pair = (text_es, text_en)
        if not text_es or not text_en or pair in seen_pairs:
            continue
        if text_es in forbidden_es or text_en in forbidden_en:
            continue
        seen_pairs.add(pair)
        output.append({"text_es": text_es, "text_en": text_en})
        if limit is not None and len(output) >= limit:
            break
    if limit is not None and len(output) < limit:
        raise ValueError(f"Solo se obtuvieron {len(output)} pares válidos de {limit} solicitados.")
    return pd.DataFrame(output, columns=["text_es", "text_en"])


def save_parallel_splits(splits):
    filenames = {"train": "train.csv", "validation": "validation.csv", "test": "test.csv"}
    for split, filename in filenames.items():
        path = TRANSLATION_DIR / filename
        splits[split].to_csv(path, index=False, encoding="utf-8")
        print(f"{split}: {len(splits[split])} pares -> {path}")

In [4]:
# 4. Descargar, limpiar y guardar Dataset A
opus = load_dataset("Helsinki-NLP/opus-100", "en-es", streaming=True)

test_df = clean_rows(opus["test"])
heldout_es = set(test_df["text_es"])
heldout_en = set(test_df["text_en"])

validation_df = clean_rows(
    opus["validation"], forbidden_es=heldout_es, forbidden_en=heldout_en
)
heldout_es.update(validation_df["text_es"])
heldout_en.update(validation_df["text_en"])

train_stream = opus["train"].shuffle(seed=SEED, buffer_size=OPUS_SHUFFLE_BUFFER)
train_df = clean_rows(
    train_stream, limit=TRAIN_SIZE, forbidden_es=heldout_es, forbidden_en=heldout_en
)

parallel_splits = {"train": train_df, "validation": validation_df, "test": test_df}
save_parallel_splits(parallel_splits)

total_pairs = sum(len(frame) for frame in parallel_splits.values())
parallel_summary = pd.DataFrame([
    {"split": split, "num_pairs": len(frame), "percentage": round(100 * len(frame) / total_pairs, 3)}
    for split, frame in parallel_splits.items()
] + [{"split": "total", "num_pairs": total_pairs, "percentage": 100.0}])
parallel_summary.to_csv(REPORT_DIR / "parallel_dataset_summary.csv", index=False, encoding="utf-8")
display(parallel_summary)
display(train_df.head())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train: 100000 pares -> /content/Proyecto-IA-Traductor/data/processed/translation/train.csv
validation: 1990 pares -> /content/Proyecto-IA-Traductor/data/processed/translation/validation.csv
test: 1999 pares -> /content/Proyecto-IA-Traductor/data/processed/translation/test.csv


,split,num_pairs,percentage
0,train,100000,96.164
1,validation,1990,1.914
2,test,1999,1.922
3,total,103989,100.000


,text_es,text_en
0,Estoy bromenado.,Just kidding.
1,En 2005/2006 se financiaron más de 50 proyecto...,"In 2005-2006, over 50 projects totalling appro..."
2,"- Sí, hola","- Yes, hello."
3,La vegetación de la zona se quema parcialmente.,The vegetation of the area is partially burnt.
4,Voy a ir a verlo.,I'm going to see it.


## Dataset B — FLEURS para evaluación Speech-to-Speech

El audio español se descarga sin `torchcodec` y se decodifica con `soundfile`. La referencia inglesa se obtiene del TSV oficial `en_us` mediante el identificador multilingüe `sentence_id`; no se genera con el traductor que será evaluado.

In [5]:
# 5. Funciones de audio y manifiestos FLEURS
FLEURS_EN_TSV_URLS = [
    f"https://huggingface.co/datasets/google/fleurs/resolve/main/data/en_us/{split}.tsv"
    for split in ("train", "dev", "test")
]
FLEURS_TSV_COLUMNS = ["sentence_id", "audio_filename", "raw_transcription", "transcription", "words", "num_samples", "gender"]


def decode_audio(audio):
    if audio.get("array") is not None and audio.get("sampling_rate") is not None:
        array = np.asarray(audio["array"], dtype=np.float32)
        sample_rate = int(audio["sampling_rate"])
    else:
        if audio.get("bytes") is not None:
            source = BytesIO(audio["bytes"])
        elif audio.get("path"):
            source = audio["path"]
        else:
            raise ValueError("Audio sin bytes, path ni array.")
        array, sample_rate = sf.read(source, dtype="float32", always_2d=False)
        array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        array = array.mean(axis=1, dtype=np.float32)
    if array.ndim != 1 or array.size == 0:
        raise ValueError("Audio vacío o con forma inválida.")
    return array, int(sample_rate)


def load_english_references():
    references = {}
    for url in FLEURS_EN_TSV_URLS:
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        frame = pd.read_csv(StringIO(response.text), sep="\t", header=None, names=FLEURS_TSV_COLUMNS, dtype={"sentence_id": str})
        for _, row in frame.iterrows():
            sentence_id = str(row["sentence_id"]).strip()
            text = str(row["raw_transcription"]).strip()
            if sentence_id and text and sentence_id not in references:
                references[sentence_id] = text
    return references


def assign_evaluation_splits(frame):
    result = frame.copy()
    indices = result.index.to_numpy(copy=True)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    n_dev = int(round(len(indices) * DEV_RATIO))
    result["split"] = "test"
    result.loc[indices[:n_dev], "split"] = "dev"
    return result

In [6]:
# 6. Descargar los 20 audios y construir Dataset B
references_en = load_english_references()
fleurs_dataset = load_dataset("google/fleurs", "es_419", streaming=True)
fleurs_stream = fleurs_dataset["train"].cast_column("audio", Audio(decode=False))
candidate_stream = fleurs_stream.shuffle(seed=SEED, buffer_size=FLEURS_SHUFFLE_BUFFER)

# Conservar únicamente audios cuya oración tenga referencia inglesa oficial.
samples = []
skipped_sentence_ids = []
for sample in candidate_stream:
    sentence_id = str(sample["id"])
    if sentence_id not in references_en:
        skipped_sentence_ids.append(sentence_id)
        continue
    samples.append(sample)
    if len(samples) == N_AUDIO_SAMPLES:
        break
if len(samples) < N_AUDIO_SAMPLES:
    raise ValueError(f"Solo se encontraron {len(samples)} audios con referencia inglesa.")
print(f"Muestras seleccionadas: {len(samples)}; candidatos omitidos sin referencia: {skipped_sentence_ids}")

rows = []
for index, sample in enumerate(samples):
    sample_id = f"fleurs_es_419_{index:04d}"
    sentence_id = str(sample["id"])
    array, sample_rate = decode_audio(sample["audio"])
    audio_path = AUDIO_DIR / f"{sample_id}.wav"
    sf.write(audio_path, array, sample_rate, subtype="FLOAT")
    rows.append({
        "sample_id": sample_id,
        "sentence_id": sentence_id,
        "speaker_id": None,
        "audio_path": audio_path.relative_to(PROJECT_ROOT).as_posix(),
        "transcription_es": str(sample.get("raw_transcription") or sample.get("transcription") or ""),
        "translation_en": references_en.get(sentence_id, ""),
        "duration_sec": round(len(array) / sample_rate, 3),
        "source": "fleurs:es_419",
    })

evaluation = assign_evaluation_splits(pd.DataFrame(rows))
missing_reference_ids = evaluation.loc[
    evaluation["translation_en"].str.strip() == "", "sentence_id"
].astype(str).tolist()
if missing_reference_ids:
    raise ValueError(f"No se encontraron referencias inglesas para sentence_id: {missing_reference_ids}")
evaluation.to_csv(MANIFEST_DIR / "evaluation_v1.csv", index=False, encoding="utf-8")
evaluation.drop(columns=["translation_en", "split"]).to_csv(
    MANIFEST_DIR / "fleurs_sample.csv", index=False, encoding="utf-8"
)
display(evaluation)

'peer closed connection without sending complete message body (received 437712813 bytes, expected 665332094)' thrown while requesting GET https://huggingface.co/datasets/google/fleurs/resolve/70bb2e84b976b7e960aa89f1c648e09c59f894dd/parquet-data/es_419/train-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].


Muestras seleccionadas: 20; candidatos omitidos sin referencia: ['466', '362']


,sample_id,sentence_id,speaker_id,audio_path,transcription_es,translation_en,duration_sec,source,split
0,fleurs_es_419_0000,35,None,data/raw/audio/fleurs_es_419_0000.wav,"Exactamente a las 8:46 a. m., la ciudad se vol...",At exactly 8:46 a.m. a hush fell across the ci...,8.88,fleurs:es_419,dev
1,fleurs_es_419_0001,1202,None,data/raw/audio/fleurs_es_419_0001.wav,"Por lo general, las hembras se relacionan estr...",The females are usually closely related to eac...,8.94,fleurs:es_419,test
2,fleurs_es_419_0002,957,None,data/raw/audio/fleurs_es_419_0002.wav,"Recuerde incluir los gastos adicionales, los i...",Don't forget to add in the extra costs of addi...,8.64,fleurs:es_419,test
3,fleurs_es_419_0003,442,None,data/raw/audio/fleurs_es_419_0003.wav,Durante el siglo XVIII Camboya sufrió varias i...,The Thais invaded Cambodia several times in th...,10.26,fleurs:es_419,dev
4,fleurs_es_419_0004,1090,None,data/raw/audio/fleurs_es_419_0004.wav,"Las manifestaciones, en ocasiones violentas, f...",The sometimes-violent protests were triggered ...,9.78,fleurs:es_419,test
5,fleurs_es_419_0005,1039,None,data/raw/audio/fleurs_es_419_0005.wav,Lo único que le digo a la gente es que ustedes...,All I say to people is you treat us the way we...,6.06,fleurs:es_419,dev
6,fleurs_es_419_0006,1163,None,data/raw/audio/fleurs_es_419_0006.wav,"En la sabana, no resulta fácil para un primate...","Out on the savanna, it is hard for a primate w...",14.82,fleurs:es_419,dev
7,fleurs_es_419_0007,476,None,data/raw/audio/fleurs_es_419_0007.wav,Cuentan que su motivación era alcanzar la fama...,"According to the story, his motivation was fam...",13.80,fleurs:es_419,dev
8,fleurs_es_419_0008,149,None,data/raw/audio/fleurs_es_419_0008.wav,La nacionalización del banco Northern Rock se ...,The Northern Rock bank had been nationalised i...,11.46,fleurs:es_419,test
9,fleurs_es_419_0009,289,None,data/raw/audio/fleurs_es_419_0009.wav,«Al instalar un circuito cerrado de televisión...,The CCTV would certainly send a strong signal ...,13.02,fleurs:es_419,dev


## Validación final

In [7]:
# 7. Validar cada fila y guardar reportes
checks = []
for _, row in evaluation.iterrows():
    path = PROJECT_ROOT / row["audio_path"]
    audio_exists = path.exists()
    audio_openable = False
    sample_rate = None
    measured_duration = None
    if audio_exists:
        try:
            info = sf.info(path)
            audio_openable = True
            sample_rate = int(info.samplerate)
            measured_duration = round(info.frames / info.samplerate, 3)
        except Exception:
            pass
    checks.append({
        "sample_id": row["sample_id"],
        "split": row["split"],
        "audio_exists": audio_exists,
        "audio_openable": audio_openable,
        "sample_rate": sample_rate,
        "duration_sec": measured_duration,
        "has_transcription": bool(str(row["transcription_es"]).strip()),
        "has_translation": bool(str(row["translation_en"]).strip()),
    })

check_df = pd.DataFrame(checks)
check_df.to_csv(REPORT_DIR / "dataset_validation.csv", index=False, encoding="utf-8")
durations = check_df["duration_sec"].dropna()
summary_rows = [
    ("num_samples", len(evaluation)),
    ("num_audio_missing", int((~check_df["audio_exists"]).sum())),
    ("num_audio_unopenable", int((~check_df["audio_openable"]).sum())),
    ("num_duplicate_ids", int(evaluation["sample_id"].duplicated(keep=False).sum())),
    ("num_missing_transcription", int((~check_df["has_transcription"]).sum())),
    ("num_missing_translation", int((~check_df["has_translation"]).sum())),
    ("splits_dev", int((evaluation["split"] == "dev").sum())),
    ("splits_test", int((evaluation["split"] == "test").sum())),
    ("num_speakers", "No disponible en FLEURS"),
    ("total_duration_sec", round(float(durations.sum()), 3)),
    ("duration_min_sec", round(float(durations.min()), 3)),
    ("duration_max_sec", round(float(durations.max()), 3)),
    ("duration_mean_sec", round(float(durations.mean()), 3)),
    ("sample_rates", "; ".join(map(str, sorted(check_df["sample_rate"].dropna().unique())))),
]
summary = pd.DataFrame(summary_rows, columns=["metric", "value"])
summary.to_csv(REPORT_DIR / "dataset_validation_summary.csv", index=False, encoding="utf-8")
display(summary)

required_zero = ["num_audio_missing", "num_audio_unopenable", "num_duplicate_ids", "num_missing_transcription", "num_missing_translation"]
values = summary.set_index("metric")["value"]
failures = {metric: values[metric] for metric in required_zero if int(values[metric]) != 0}
if failures:
    raise AssertionError(f"Falló la validación final: {failures}")
print("VALIDACIÓN COMPLETA: no faltan audios, textos ni traducciones.")

,metric,value
0,num_samples,20
1,num_audio_missing,0
2,num_audio_unopenable,0
3,num_duplicate_ids,0
4,num_missing_transcription,0
5,num_missing_translation,0
6,splits_dev,14
7,splits_test,6
8,num_speakers,No disponible en FLEURS
9,total_duration_sec,203.34


VALIDACIÓN COMPLETA: no faltan audios, textos ni traducciones.


## Empaquetar y descargar resultados

Los datos se generan dentro de la copia del repositorio en Colab. Esta celda crea un ZIP únicamente con los artefactos —sin incluir `.git` ni el código— y abre su descarga directa.

In [8]:
# 8. Comprimir únicamente los artefactos generados (no incluye .git)
archive_path = Path("/content/IA-Proyecto_dataset_artifacts.zip") if IN_COLAB else PROJECT_ROOT.parent / "IA-Proyecto_dataset_artifacts.zip"
artifact_roots = [TRANSLATION_DIR, MANIFEST_DIR, AUDIO_DIR, REPORT_DIR]
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact_root in artifact_roots:
        for source_path in artifact_root.rglob("*"):
            if source_path.is_file():
                archive.write(source_path, source_path.relative_to(PROJECT_ROOT))
print("Archivo ZIP creado:", archive_path, f"({archive_path.stat().st_size / 1024**2:.2f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))
else:
    print("Resultados locales en:", archive_path)

Archivo ZIP creado: /content/IA-Proyecto_dataset_artifacts.zip (14.96 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Archivos generados

```text
data/processed/translation/train.csv
data/processed/translation/validation.csv
data/processed/translation/test.csv
data/processed/manifests/fleurs_sample.csv
data/processed/manifests/evaluation_v1.csv
data/raw/audio/*.wav
outputs/dataset/parallel_dataset_summary.csv
outputs/dataset/dataset_validation.csv
outputs/dataset/dataset_validation_summary.csv
```